# Running Immune in production

Immune is built to sit in the request path of a production app: it fails open, bounds its own latency, masks what it
sends out, and shares state across workers. This guide covers what to set before real traffic flows.

## Checklist

| Setting | Why |
| --- | --- |
| `TYPESAFE_API_KEY` from your secret store | Without Jev only floor candidates act |
| Sites named with `immune.site(...)` and pinned in `immune.yaml` | Stable profiles and per-site posture |
| Tool capabilities declared | Precise checks on tool calls |
| `state: {backend: redis}` or `IMMUNE_STATE_URL` with several workers | Sessions, provenance, confirmations and promotions shared |
| `sensor.api_keys` and `sensor.requests_per_minute` sized for your traffic | Jev capacity under load |
| `sensor.on_outage` chosen | What floor candidates do while Jev is unreachable |
| `on_internal_error` chosen | Fail open (default) or refuse |
| Logs, `privacy.verdict_log` or OpenTelemetry wired up | You can see what Immune does |
| `immune doctor --live` in your deploy checks | Catches a missing key or blocked network early |
| A week in `auto` reading `would_action`, then per-site `enforce` | Enforce what fits your traffic |

## 1. Several workers share one state

Immune keeps sessions (taint, provenance, confirmations), site profiles and promotions in its state. With one
process, local files are fine. With several workers, or several machines, share it:

| Backend | Configure | When |
| --- | --- | --- |
| `local` (default) | Nothing; files under `IMMUNE_HOME` | One process |
| `sqlite` | `state: {backend: sqlite, path: /var/lib/app/immune.db}` or `IMMUNE_STATE_URL=sqlite:///…` | Several workers on one host |
| `redis` | `state: {backend: redis, url: redis://…}` or `IMMUNE_STATE_URL=redis://…` | Several hosts |

If the shared backend becomes unreachable, Immune fails over to memory and retries after `failover_cooldown_s`.

Two runtimes stand in for two workers here. The first reads a document in a customer's session; the second serves the
next message in the same session and knows the session has read outside content:

In [1]:
import tempfile
from pathlib import Path

import immune
from immune.testing import FakeReply, ImmuneHarness

shared = Path(tempfile.mkdtemp()) / "immune.db"
config = {"state": {"backend": "sqlite", "path": str(shared)}}
article = immune.untrusted("Returns: unused items within 30 days with a receipt.", source="kb:returns")
SYSTEM = {"role": "system", "content": "You answer questions for Pinecrest Outfitters customers."}

worker_a = ImmuneHarness(Path(tempfile.mkdtemp()), config=config, script=FakeReply("Within 30 days."))
with immune.session("customer-9"):
    first = worker_a.openai().chat.completions.create(
        model="gpt-5.4-mini", messages=[SYSTEM, {"role": "user", "content": f"Return window?\n\n{article}"}]
    )
print("worker A:", worker_a.verdict(first).taint)
worker_a.close()

worker_b = ImmuneHarness(Path(tempfile.mkdtemp()), config=config, script=FakeReply("You're welcome!"))
with immune.session("customer-9"):
    second = worker_b.openai().chat.completions.create(
        model="gpt-5.4-mini", messages=[SYSTEM, {"role": "user", "content": "Thanks!"}]
    )
print("worker B:", worker_b.verdict(second).taint, "(same session, different worker, no document in this call)")
worker_b.close()

worker A: external
worker B: external (same session, different worker, no document in this call)


## 2. Jev capacity and latency

| Setting | Default | Meaning |
| --- | --- | --- |
| `sensor.timeout_ms` | `800` | Per Jev request. A miss continues without Jev |
| `sensor.deadline_margin_ms` | `250` | Extra time for screening as a whole |
| `sensor.requests_per_minute` | `1200` | Per key. The quota governor spreads requests over keys |
| `sensor.api_keys` | `[]` | A pool of keys; `TYPESAFE_API_KEY` is used when empty |
| `sensor.coalesce` | `false` | Merge the input and first data request into one |
| `sensor.breaker_*` | `0.2`, `60`, `30` | Stop calling a failing Jev for the cooldown, instead of adding latency to every call |

Under pressure, the governor sheds the least important questions first (those for observed detectors), and floor
questions wait briefly for capacity.

**How much latency does Immune add?** Input screening runs in parallel with your model call, so it is usually hidden.
Output screening adds one Jev round trip after the model finishes. Measured on this machine:

In [2]:
import statistics
import time

from openai import OpenAI

import immune

SYSTEM = {
    "role": "system",
    "content": "You are the support assistant for Pinecrest Outfitters. Answer in one sentence.",
}
QUESTIONS = [
    "Do you sell headlamps?",
    "Do you sell tent stakes?",
    "Do you sell water filters?",
    "Do you sell bear spray?",
]


def timed(client: OpenAI, question: str):
    started = time.perf_counter()
    response = client.chat.completions.create(
        model="gpt-5.4-mini", messages=[SYSTEM, {"role": "user", "content": question}]
    )
    return (time.perf_counter() - started) * 1000, response


without = [timed(OpenAI(), question)[0] for question in QUESTIONS[1:]]
immune.init()
with immune.site("latency-check"):
    timed(OpenAI(), QUESTIONS[0])  # the first call on a new site also profiles it
    measured = [timed(OpenAI(), question) for question in QUESTIONS[1:]]
verdict = immune.verdict(measured[-1][1])
print(f"median without Immune: {statistics.median(without):6.0f} ms")
print(f"median with Immune:    {statistics.median(ms for ms, _ in measured):6.0f} ms")
sensor = verdict.sensor
print(f"Jev for the last call: {sensor.calls} requests, {sensor.input_tokens} tokens, {sensor.latency_ms:.0f} ms")

median without Immune:    921 ms
median with Immune:      1009 ms
Jev for the last call: 2 requests, 1505 tokens, 256 ms


Numbers vary with your network, model and region; `immune doctor --live` measures a real round trip, and
`python bench/run.py` (in the repository) measures Immune's own overhead, a few milliseconds per call. Identical
requests are answered from a short-lived cache, so a verdict can show `0` Jev requests.

## 3. When Jev is unreachable

Missing keys, network trouble, an open circuit breaker or a missed deadline all mean Immune continues without Jev.
Your call still goes through, and `sensor.on_outage` decides what the reflexes' candidates do:

| `sensor.on_outage` | Floor candidates (keys, data-carrying links, off-list destinations, loops…) | Everything else |
| --- | --- | --- |
| `act_on_candidates` (default) | Acted on without Jev; the evidence says so | Waits for Jev |
| `pass` | Let through | Waits for Jev |

A failing sensor stands in for an outage here:

In [3]:
import tempfile
from pathlib import Path

from immune.testing import FakeReply, ImmuneHarness, MockSensor

reply = FakeReply("Your config: bucket=orders-prod, access_key_id=AKIA2E5QZ7XJ4M8RW3TD")
for choice in ["act_on_candidates", "pass"]:
    harness = ImmuneHarness(
        Path(tempfile.mkdtemp()),
        sensor=MockSensor(fail=True),
        script=reply,
        config={"sensor": {"on_outage": choice}},
    )
    response = harness.openai().chat.completions.create(
        model="gpt-5.4-mini", messages=[{"role": "user", "content": "Show me the prod config."}]
    )
    verdict = harness.verdict(response)
    print(f"on_outage={choice}: {response.choices[0].message.content}")
    print(f"    sensor={verdict.sensor.name}  evidence={[list(hit.evidence) for hit in verdict.hits]}")
    harness.close()

immune: sensor unavailable for input (mock sensor configured to fail); floor candidates act without Jev


immune: sensor unavailable for outbound screening; floor candidates act without Jev


on_outage=act_on_candidates: Your config: bucket=orders-prod, access_key_id=[redacted]
    sensor=tier0_only  evidence=[['secret:aws_access_key', 'acted without Jev: the sensor was unavailable (sensor.on_outage: act_on_candidates)']]
on_outage=pass: Your config: bucket=orders-prod, access_key_id=AKIA2E5QZ7XJ4M8RW3TD
    sensor=tier0_only  evidence=[]


Immune logs a warning (`immune: sensor unavailable …`) for each call screened without Jev, throttled so an outage
doesn't flood your logs.

## 4. When Immune itself fails

Any exception inside Immune passes the original call through with a warning (`on_internal_error: pass`, the
default), so a bug in Immune never takes your app down. Set `on_internal_error: block` for applications where an
unscreened reply is worse than no reply. The verdict of a passed-through call says so in its explanation.

## 5. What leaves your process

To screen a call, Immune sends TypeSafe the relevant text with personal data and secrets replaced by typed
placeholders first (`privacy.redact_before_sensor: true`). Here a scripted Jev records exactly what it was sent:

In [4]:
import json
import tempfile
from pathlib import Path

from immune.testing import FakeReply, ImmuneHarness, MockSensor

sensor = MockSensor()
harness = ImmuneHarness(Path(tempfile.mkdtemp()), sensor=sensor, script=FakeReply("I've updated your details."))
harness.openai().chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": "You are the support assistant for Pinecrest Outfitters."},
        {"role": "user", "content": "Update my email to jordan.lee@example.com and charge card 4111 1111 1111 1111."},
    ],
)
sent = json.dumps([state for state, _ in sensor.calls])
print("raw email sent to Jev:", "jordan.lee@example.com" in sent)
print("raw card sent to Jev: ", "4111 1111 1111 1111" in sent)
print("placeholders sent:    ", sorted({token for token in ("[EMAIL]", "[CARD]") if token in sent}))
harness.close()

raw email sent to Jev: False
raw card sent to Jev:  False
placeholders sent:     ['[CARD]', '[EMAIL]']


| Setting | Effect |
| --- | --- |
| `privacy.redact_before_sensor: true` (default) | Mask personal data and secrets before Jev |
| `privacy.profiling: local` | Never send your system prompt; profile sites from local heuristics instead |
| `privacy.log` | How much evidence logs keep (`redacted` by default) |
| `telemetry.langsmith.inputs: none` | Send no text to LangSmith |

Nothing is ever sent to the Immune project itself.

## 6. Switches for incidents

| To… | Do | Effect |
| --- | --- | --- |
| Stop enforcing one detector at one site | `immune.configure(sites={"s": {"observe": ["threat"]}})` | Next call |
| Stop screening one site | `immune.configure(sites={"s": {"enabled": False}})` | Next call |
| Observe everything | `immune.configure(mode="observe")` or `IMMUNE_MODE=observe` | Next call / next deploy |
| Remove Immune without a code change | `IMMUNE_DISABLED=1` | Next start |
| Remove Immune from a running process | `immune.shutdown()` | Immediately |

## 7. Where `immune.init()` goes

```python
# FastAPI
from contextlib import asynccontextmanager
import immune

@asynccontextmanager
async def lifespan(app):
    immune.init()          # reads ./immune.yaml, or IMMUNE_CONFIG
    yield
    immune.shutdown()

app = FastAPI(lifespan=lifespan)
```

```python
# Gunicorn with preload_app = True: one runtime per worker
def post_fork(server, worker):
    import immune
    immune.init()
```

```python
# Celery: one runtime per worker process
from celery.signals import worker_process_init

@worker_process_init.connect
def start_immune(**_):
    import immune
    immune.init()
```

## Recap

- Share state with SQLite or Redis when you run more than one worker
- Size Jev capacity with `api_keys` and `requests_per_minute`; latency is bounded by `timeout_ms`
- Outages keep floor candidates acting by default; internal errors fail open
- Personal data and secrets are masked before anything leaves your process
- Every posture change is available live, without a deploy

Next: [Troubleshooting and FAQ](16_troubleshooting_and_faq.ipynb)